# ポケモンカード AI 自由研究 — Colab で追加学習する

このノートブックは、家のパソコンで遊んだ対戦の記録（Google ドライブに保存されたもの）を使って、
公開ずみのモデル `yoheikobashi/ptcg-dusknoir-deberta-reranker` を**追加学習**します。

研究テーマ: **ogerpon_mono（オーガポン）に対する勝率をどこまで上げられるか。**

やること（上から順にセルを実行するだけ）:

1. 設定（学習に使う期間・学習率・GPU の種類）
2. Google ドライブをつなぐ／リポジトリとゲームエンジンの準備
3. **学習前に対戦履歴を集計**（何試合あるか、勝率はどうか）
4. 対戦履歴 → 学習データ に変換
5. 追加学習（T4 / L4 / A100 を自動で切り替え）
6. 学習したモデルをドライブに保存
7. （おすすめ）ヒューリスティックエンジンと対戦させて、学習前後を比べる

> ⚠️ 学習の前に必ず「ランタイム → ランタイムのタイプを変更 → GPU」を選んでください。
> 無料版では T4 が割り当てられます（それで動きます）。


## 1. 設定

ここだけ書き換えれば、あとは実行するだけです。

In [ ]:
# @title 設定 { display-mode: "form" }
DRIVE_DIR = "/content/drive/MyDrive/PTCG"  # @param {type:"string"}
# 学習に使う対戦の期間（空にすると全部）。例: "2026-08-16"
SINCE = ""  # @param {type:"string"}
UNTIL = ""  # @param {type:"string"}
# 学習に使う相手デッキ。"" なら全部、"ogerpon_mono" ならオーガポン戦だけ
OPPONENT = ""  # @param ["", "ogerpon_mono", "alakazam_nz", "marnie_grimmsnarl", "mega_abomasnow_sample"] {allow-input: true}
# 学習率。5e-6 が標準。上げすぎると「前に覚えたこと」を壊します
LEARNING_RATE = 5e-6  # @param [1e-6, 2e-6, 5e-6, 1e-5, 2e-5] {type:"raw", allow-input: true}
EPOCHS = 2.0  # @param {type:"number"}
# 元のモデルからどれだけ離れてよいか（大きいほど元のまま。0 で自由）
L2SP = 1e-3  # @param [0, 1e-4, 1e-3, 1e-2] {type:"raw", allow-input: true}
# GPU の種類。auto なら割り当てられた GPU を見て自動で決めます
GPU_PROFILE = "auto"  # @param ["auto", "T4", "L4/A100", "CPU"]
# 保存するモデルの名前（ドライブの models/ の下にこの名前で入ります）
MODEL_NAME = "kenkyu_r1"  # @param {type:"string"}
BASE_MODEL = "yoheikobashi/ptcg-dusknoir-deberta-reranker"  # @param {type:"string"}
HUMAN_DECK = "dragapult_dusknoir"  # @param {type:"string"}

LOGS_DIR = DRIVE_DIR + "/logs"
MODELS_DIR = DRIVE_DIR + "/models"
OUT_DIR = MODELS_DIR + "/" + MODEL_NAME
RESULTS = DRIVE_DIR + "/kenkyu_results.json"
print("logs   :", LOGS_DIR)
print("out    :", OUT_DIR)

## 2. 準備（ドライブ・リポジトリ・ゲームエンジン）

リポジトリとモデルは公開なので、ここでのログインは Google ドライブだけです。
`cg`（対戦エンジン）はコンペのファイルなので GitHub には入っていません。
家のパソコンで `setup_local.py --drive` を実行してあれば、ドライブの `cg/` にコピーが置かれているので、
それをそのまま使います。無ければ Kaggle からダウンロードします（`DRIVE_DIR/kaggle.json` が必要）。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, subprocess, sys, zipfile

REPO = "/content/pokemon-card-bert"
GITHUB = "https://github.com/yohei-kobashi/pokemon-card-bert.git"

def have_repo():
    return os.path.isfile(REPO + "/play_server.py")

if not have_repo():
    # リポジトリは公開なので、そのままクローンできます（ログイン不要）。
    # ネットワークや GitHub 側の都合で取れなかったときだけ、家のPCの
    # setup_local.py --drive がドライブに置いた repo.zip から展開します。
    subprocess.run(["git", "clone", "-q", GITHUB, REPO])
if not have_repo():
    zpath = DRIVE_DIR + "/repo.zip"
    if not os.path.exists(zpath):
        raise SystemExit("リポジトリを取得できません。ネットワークを確認するか、家のPCで\n"
                         "  python tools/kenkyu/setup_local.py --drive \"<ドライブ>\"\n"
                         "を実行して repo.zip を作ってから、もう一度実行してください")
    os.makedirs(REPO, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(REPO)
    print("repo.zip から展開しました")
elif os.path.isdir(REPO + "/.git"):
    subprocess.run(["git", "-C", REPO, "pull", "-q"])

os.chdir(REPO)
sys.path[:0] = [REPO, REPO + "/cg-lib", REPO + "/tools"]

# Colab には torch が入っています。足りないものだけ
!pip -q install "transformers>=4.44" "tokenizers>=0.19" huggingface_hub
print("repo:", REPO, "| play_server.py:", os.path.exists("play_server.py"))

In [ ]:
# ゲームエンジンを置く（ドライブのコピー優先 → 無ければ Kaggle から）
drive_cg = DRIVE_DIR + "/cg"
if os.path.isdir(drive_cg):
    cmd = f'python tools/kenkyu/setup_local.py --from-dir "{drive_cg}"'
else:
    kj = DRIVE_DIR + "/kaggle.json"
    if os.path.exists(kj):
        os.makedirs("/root/.kaggle", exist_ok=True)
        shutil.copy(kj, "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
        !pip -q install kaggle
    cmd = 'python tools/kenkyu/setup_local.py'
print(cmd)
!{cmd}

## 3. 学習前に対戦履歴を集計する

「何試合たまったか」「相手ごとの勝率」「日ごとの勝率」を出します。
研究のグラフはここの数字を使うと書きやすいです。
`SINCE` / `UNTIL` を変えると、**学習に使うのと同じ範囲**の集計になります。

In [ ]:
# ドライブに zip でアップロードした場合はここで展開します（Drive アプリが無い PC 向け）
import glob, os
os.makedirs(LOGS_DIR, exist_ok=True)
zips = glob.glob(DRIVE_DIR + "/*.zip") + glob.glob(LOGS_DIR + "/*.zip")
for z in zips:
    print("展開:", z)
    !unzip -n -q "{z}" -d "{LOGS_DIR}"
print("ログ:", len(glob.glob(LOGS_DIR + "/*.json*")), "ファイル")

In [ ]:
args = ""
if SINCE: args += f' --since "{SINCE}"'
if UNTIL: args += f' --until "{UNTIL}"'
if OPPONENT: args += f' --opp {OPPONENT}'

cmd = f'python tools/kenkyu/log_stats.py --logs "{LOGS_DIR}"{args} --json /content/stats.json'
print(cmd)
!{cmd}

## 4. 対戦履歴 → 学習データ

自分（人間）が選んだ手を「正解」として、その場面で選べた手ぜんぶと一緒に並べたものが 1 行になります。
勝った試合の手は少し強めに、負けた試合の手は少し弱めに重みをつけます
（負けた試合でも、ほとんどの手は正しいからです）。

In [ ]:
ROWS = "/content/human_rows.jsonl.gz"
cmd = f'python tools/human_rows.py --logs "{LOGS_DIR}"{args} --deck {HUMAN_DECK} --out {ROWS}'
print(cmd)
!{cmd}

import gzip
n = sum(1 for _ in gzip.open(ROWS, "rt"))
print("学習データ:", n, "行")
if n < 200:
    print("⚠️ 少なすぎます。もう少し対戦を増やしてから学習しましょう（目安 500 行＝約 15 試合）")

## 5. 追加学習

GPU によって計算のしかたを変えます。

| GPU | 精度 | 理由 |
|---|---|---|
| T4（無料版） | fp16 + ロススケーリング | T4（Turing）には bf16 の演算器が無く、bf16 を選ぶとエミュレーションで**遅くなるだけ**。fp16 は勾配が小さすぎて 0 に潰れるので、ロススケーリングを自動で入れます |
| L4 / A100 | bf16 | 数値の範囲が広く、ロススケーリング無しで安定 |
| CPU | fp32 | GPU が無いときの最後の手段（とても遅い） |

`--max-minutes` を付けているので、時間切れになっても**そこまでの結果は必ず保存**されます。

In [ ]:
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
mem = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print("GPU:", name, "%.1f GiB" % mem)

if GPU_PROFILE == "auto":
    profile = "CPU" if not torch.cuda.is_available() else (
        "L4/A100" if torch.cuda.is_bf16_supported() else "T4")
else:
    profile = GPU_PROFILE
PRESET = {
    "T4":      dict(precision="fp16", maxlen=448, accum=8, minutes=120),
    "L4/A100": dict(precision="bf16", maxlen=512, accum=8, minutes=240),
    "CPU":     dict(precision="fp32", maxlen=384, accum=8, minutes=30),
}[profile]
print("プロファイル:", profile, PRESET)

In [ ]:
cmd = (f'python tools/dusk_plan_train.py --data {ROWS} --model {BASE_MODEL} '
       f'--out /content/out_model --lr {LEARNING_RATE} --epochs {EPOCHS} --l2sp {L2SP} '
       f'--precision {PRESET["precision"]} --maxlen {PRESET["maxlen"]} '
       f'--accum {PRESET["accum"]} --max-minutes {PRESET["minutes"]}')
print(cmd)
!{cmd}

`plan-conformance before / after` は「その場面で人間と同じ手を選べた割合」です。
after が before より上がっていれば、人間のプレイを覚えたということ。
ただし**それは勝率が上がったという意味ではありません**（次のセルで実際に対戦させて確かめます）。

## 6. ドライブに保存

In [ ]:
import shutil, os
os.makedirs(MODELS_DIR, exist_ok=True)
if os.path.isdir(OUT_DIR):
    shutil.rmtree(OUT_DIR)
shutil.copytree("/content/out_model", OUT_DIR)
print("保存しました:", OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))

## 7. 強くなったか確かめる（ogerpon_mono との対戦）

同じ相手・同じデッキ・先攻後攻を交互にして戦わせ、勝率を記録します。
結果はドライブの `kenkyu_results.json` にたまっていくので、
**あとから何個でもモデルを比べられます**（家のパソコンで測っても同じ表に入ります）。

40 試合で ±15 ポイントくらいのブレがあります。**「95% CI」が重なっている 2 つは、まだ差があるとは言えません。**
本気で比べるなら 200 試合以上にしてください（GPU なら 1 試合あたり数秒〜十数秒、CPU だと 1〜3 分かかります。
モデルを使わないヒューリスティックは 1 試合 0.2 秒なので何試合でも回せます）。
同じ `--tag` でもう一度回すと、表では**合算**されます。

In [ ]:
GAMES = 40  # @param {type:"integer"}

EV = "python -u tools/kenkyu/battle_eval.py"   # -u = 1試合ごとに結果が流れる
# 比べる相手その1: モデルなしのヒューリスティックエンジン（速いので多めに回す）
cmd = f'{EV} --model engine --games 200 --tag heuristic --results "{RESULTS}" --quiet'
print(cmd); get_ipython().system(cmd)

# 比べる相手その2: 学習まえの公開モデル
cmd = f'{EV} --model {BASE_MODEL} --games {GAMES} --tag base --device cuda --results "{RESULTS}"'
print(cmd); get_ipython().system(cmd)

# 今回学習したモデル
cmd = f'{EV} --model "{OUT_DIR}" --games {GAMES} --tag {MODEL_NAME} --device cuda --results "{RESULTS}"'
print(cmd); get_ipython().system(cmd)

In [ ]:
cmd = f'python tools/kenkyu/battle_eval.py --compare --baseline base --results "{RESULTS}"'
!{cmd}

## 8. 学習したモデルを家のパソコンで使う

ドライブの `models/<名前>` フォルダを家のパソコンにダウンロードして、そのパスを指定します。

```bash
# 人間と対戦させる（play_server の /manage で ai_agent = lm_dusknoir を選ぶ）
PTCG_MODEL=/path/to/kenkyu_r1 python play_server.py

# エンジンと対戦させて勝率を測る（--model にパスを渡すので環境変数は不要）
python tools/kenkyu/battle_eval.py --model /path/to/kenkyu_r1 --games 40 --tag kenkyu_r1
python tools/kenkyu/battle_eval.py --compare --baseline base
```

### 次に試すとよいこと

- **学習率を変える**（1e-6 / 5e-6 / 2e-5）。上げすぎると勝率が下がります — それも立派な結果です。
- **期間で切る**（`SINCE`）。上手くなってからの試合だけで学習すると強くなるか？
- **相手で切る**（`OPPONENT = "ogerpon_mono"`）。オーガポン戦だけで学習した方が、
  オーガポンに強くなるか？ 逆に他の相手に弱くなっていないかも測れます
  （`battle_eval.py --opp alakazam_nz` など）。
- **試合数を増やす**。行数と勝率の関係をグラフにすると、研究らしいまとめになります。
- **`L2SP` を 0 にしてみる**。元のモデルから離れてよいとどうなるか。
